# Model Training + MLflow Tracking
This notebook trains models and logs experiments using MLflow.

In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path
from mlflow.tracking import MlflowClient
import joblib
from datetime import datetime

In [2]:
trial_number = "_"+datetime.now().strftime("%Y%m%d_%H%M%S")
print("Experiment started at: ", trial_number)

Experiment started at:  _20260502_173135


In [3]:
df = pd.read_csv('../data/processed/heart_clean.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [4]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
categorical_cols = ['cp', 'restecg', 'slope', 'thal']
numeric_cols = [col for col in X.columns if col not in categorical_cols]
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

In [6]:
def evaluate(model):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    return {
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'roc_auc': roc_auc_score(y_test, probs),
        'confusion_matrix': confusion_matrix(y_test, preds)
    }

In [7]:
# -----------------------------
# Helper: Format run name
# -----------------------------
def format_run_name(model_name, params):
    param_str = "_".join(
        [f"{k.split('__')[-1]}={v}" for k, v in params.items()]
    )
    return f"{model_name}_{param_str}"

In [8]:
tracking_path = Path("../mlruns").resolve()
mlflow.set_tracking_uri(f"file:///{tracking_path.as_posix()}")

# -----------------------------
# Models & Params
# -----------------------------
models = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "random_forest": RandomForestClassifier()
}

param_grid = {
    "logistic_regression": {
        "model__C": [0.1, 1, 10]
    },
    "random_forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [5, 10]
    }
}

# -----------------------------
# Training + ML Flow Logging
# -----------------------------
for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    grid = GridSearchCV(
        pipeline,
        param_grid[name],
        cv=3,
        scoring="roc_auc",
        return_train_score=True
    )

    grid.fit(X_train, y_train)

    results = grid.cv_results_

    # -----------------------------
    # Log ALL hyperparameter runs
    # -----------------------------
    mlflow.set_experiment("heart_disease_cross_validation_runs"+str(trial_number))
    for i in range(len(results["params"])):

        params = results["params"][i]
        run_name = format_run_name(name, params)

        with mlflow.start_run(run_name=run_name):

            mlflow.set_tag("model_type", name)

            # Log params
            mlflow.log_params(params)

            # Log CV metrics
            mlflow.log_metric("mean_test_score", results["mean_test_score"][i])
            mlflow.log_metric("std_test_score", results["std_test_score"][i])

            if "mean_train_score" in results:
                mlflow.log_metric("mean_train_score", results["mean_train_score"][i])

    # -----------------------------
    # Log BEST model separately
    # -----------------------------
    mlflow.set_experiment("heart_disease_best_model_runs"+str(trial_number))
    best_model = grid.best_estimator_

    with mlflow.start_run(run_name=f"{name}_BEST"):

        mlflow.set_tag("model_type", name)
        mlflow.set_tag("stage", "best_model")

        metrics = evaluate(best_model)

        # Log best params
        mlflow.log_params(grid.best_params_)

        # Log test metrics
        mlflow.log_metric("test_accuracy", metrics["accuracy"])
        mlflow.log_metric("test_precision", metrics["precision"])
        mlflow.log_metric("test_recall", metrics["recall"])
        mlflow.log_metric("test_roc_auc", metrics["roc_auc"])

        # Log model
        mlflow.sklearn.log_model(best_model, "model")

        print(f"{name} BEST → {metrics}")

2026/05/02 17:31:46 INFO mlflow.tracking.fluent: Experiment with name 'heart_disease_cross_validation_runs_20260502_173135' does not exist. Creating a new experiment.
2026/05/02 17:31:47 INFO mlflow.tracking.fluent: Experiment with name 'heart_disease_best_model_runs_20260502_173135' does not exist. Creating a new experiment.


logistic_regression BEST → {'accuracy': 0.8524590163934426, 'precision': 0.8484848484848485, 'recall': 0.875, 'roc_auc': 0.9245689655172413, 'confusion_matrix': array([[24,  5],
       [ 4, 28]], dtype=int64)}


c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\MLOps\MLOps Assignment\MLOps-Assignment\mlops_env\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\MLOps\MLOps Assignment\MLOps-Assignment\mlops_env\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


random_forest BEST → {'accuracy': 0.8852459016393442, 'precision': 0.9310344827586207, 'recall': 0.84375, 'roc_auc': 0.9504310344827587, 'confusion_matrix': array([[27,  2],
       [ 5, 27]], dtype=int64)}


## Running MLflow UI separately:
mlflow ui
Open http://127.0.0.1:5000

In [9]:

# -------------------------------------------------------
# Compare best LR vs best RF → Register the winner
# -------------------------------------------------------

client = MlflowClient()

# Fetch all runs from the best-model experiment, ranked by roc_auc
experiment = client.get_experiment_by_name("heart_disease_best_model_runs"+str(trial_number))
best_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_roc_auc DESC"]
)

# The top run is the overall winner
winner = best_runs[0]

model_type = winner.data.tags.get("model_type", "unknown")
params     = winner.data.params
run_id     = winner.info.run_id
roc_auc    = winner.data.metrics.get("test_roc_auc")


registered_name = "heart_disease_pred_model"

print(f"Winner  : {model_type}")
print(f"Run ID  : {run_id}")
print(f"ROC-AUC : {roc_auc:.4f}")
print(f"Params  : {params}")
print(f"Registering as → '{registered_name}'")

# ── MLflow Model Registry ──────────────────────────────
model_uri = f"runs:/{run_id}/model"
mv = mlflow.register_model(model_uri=model_uri, name=registered_name)

print(f"\nMLflow registration successful!")
print(f"  Name    : {mv.name}")
print(f"  Version : {mv.version}")
print(f"  Stage   : {mv.current_stage}")

# ── Save .pkl to models/ folder ────────────────────────
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

pkl_filename = f"{registered_name}.pkl"
pkl_path = models_dir / pkl_filename

# Load the logged model artifact from MLflow and persist it
winner_model = mlflow.sklearn.load_model(model_uri)
joblib.dump(winner_model, pkl_path)

print(f"\nModel saved to : {pkl_path.resolve()}")


Winner  : random_forest
Run ID  : 1377e9fb388744ccbe43c5efa2a660d3
ROC-AUC : 0.9504
Params  : {'model__max_depth': '5', 'model__n_estimators': '200'}
Registering as → 'heart_disease_pred_model'

MLflow registration successful!
  Name    : heart_disease_pred_model
  Version : 1
  Stage   : None


Successfully registered model 'heart_disease_pred_model'.
Created version '1' of model 'heart_disease_pred_model'.



Model saved to : C:\Bhooshaan_Local\BITS Local\Sem2\Assignments\MLOps\MLOps Assignment\MLOps-Assignment\models\heart_disease_pred_model.pkl
